In [1]:
#============================================
# Celda 1 — Imports y carga del JSON
#============================================
import json, glob, pandas as pd, numpy as np, os
from pathlib import Path
from datetime import datetime

# ROOT dinámico — igual que en notebooks de insights
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError(f"No se encontró requirements.txt subiendo desde {Path.cwd()}")
os.chdir(ROOT)
print(f"✅ ROOT: {ROOT}")

def ultimo_json(carpeta_base: str) -> str:
    archivos = glob.glob(f'{carpeta_base}/**/*.json', recursive=True)
    assert len(archivos) > 0, f"❌ No hay JSONs en {carpeta_base}"
    def parse_fecha(ruta):
        try:
            return datetime.strptime(Path(ruta).stem, "%d_%m_%Y")
        except ValueError:
            return datetime.min
    return max(archivos, key=parse_fecha)

# ── ambiental — rutas desde raíz del repo ──
f = ultimo_json('data/ambiental')       # ← sin ../../
data = json.load(open(f))
print(f'Archivo cargado: {f}')
print(f'Keys disponibles: {list(data.keys())}')

# Validar estructura
assert 'observacion_meteorologica' in data, \
    f"❌ Key 'observacion_meteorologica' no encontrada. Keys: {list(data.keys())}"
assert 'estaciones' in data['observacion_meteorologica'], \
    (f"❌ Key 'estaciones' no encontrada.\n"
     f"Keys en 'observacion_meteorologica': {list(data['observacion_meteorologica'].keys())}\n"
     f"La API AEMET probablemente devolvió error en la última captura.")
assert 'calidad_aire' in data, \
    f"❌ Key 'calidad_aire' no encontrada. Keys: {list(data.keys())}"

print(f"✅ Estructura JSON validada")
print(f"   Estaciones AEMET: {len(data['observacion_meteorologica']['estaciones'])}")
print(f"   Ciudades WAQI:    {len(data['calidad_aire'].get('ciudades', []))}")

✅ ROOT: /workspaces/TFG_Spain-s_Migratory_Flow
Archivo cargado: data/ambiental/2026/06/06_06_2026.json
Keys disponibles: ['timestamp', 'observacion_meteorologica', 'calidad_aire']
✅ Estructura JSON validada
   Estaciones AEMET: 10253
   Ciudades WAQI:    30


In [2]:
#============================================
# Celda 2 — Explorar estructura AEMET
#============================================
aemet_raw  = data['observacion_meteorologica']
estaciones = aemet_raw['estaciones']

print(f'Total estaciones: {aemet_raw["total_estaciones"]}')
print(f'Timestamp captura: {aemet_raw["timestamp_captura"]}')
print(f'\nKeys de una estación:')
print(list(estaciones[0].keys()))
print(f'\nEjemplo estación[0]:')
print(json.dumps(estaciones[0], indent=2, ensure_ascii=False))


Total estaciones: 10253
Timestamp captura: 2026-06-06T11:46:45.416039

Keys de una estación:
['idema', 'ubi', 'lat', 'lon', 'alt', 'fint', 'ta', 'tamin', 'tamax', 'prec', 'hr', 'vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar', 'vis', 'inso', 'ts', 'tpr']

Ejemplo estación[0]:
{
  "idema": "0009X",
  "ubi": "ALFORJA",
  "lat": 41.213892,
  "lon": 0.963335,
  "alt": 406.0,
  "fint": "2026-06-05T23:00:00+0000",
  "ta": 15.7,
  "tamin": 15.7,
  "tamax": 16.0,
  "prec": 0.0,
  "hr": 87.0,
  "vv": 0.6,
  "vmax": 1.8,
  "dv": 49.0,
  "dmax": 20.0,
  "pres": null,
  "pres_nmar": null,
  "vis": null,
  "inso": null,
  "ts": null,
  "tpr": null
}


In [3]:
#============================================
# Celda 3 — Parsear AEMET a DataFrame
#============================================
df_aemet = pd.DataFrame(estaciones)

# Convertir timestamp
df_aemet['fint'] = pd.to_datetime(df_aemet['fint'], utc=True, errors='coerce')
df_aemet['fecha'] = df_aemet['fint'].dt.date

# Columnas numéricas
cols_num = ['lat','lon','alt','ta','tamin','tamax','prec','hr',
            'vv','vmax','dv','dmax','pres','pres_nmar','vis','inso','ts','tpr']
for c in cols_num:
    if c in df_aemet.columns:
        df_aemet[c] = pd.to_numeric(df_aemet[c], errors='coerce')

print(f'Shape: {df_aemet.shape}')
print(f'Columnas: {list(df_aemet.columns)}')
df_aemet.head()


Shape: (10253, 22)
Columnas: ['idema', 'ubi', 'lat', 'lon', 'alt', 'fint', 'ta', 'tamin', 'tamax', 'prec', 'hr', 'vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar', 'vis', 'inso', 'ts', 'tpr', 'fecha']


,idema,ubi,lat,lon,alt,fint,ta,tamin,tamax,prec,...,vmax,dv,dmax,pres,pres_nmar,vis,inso,ts,tpr,fecha
0,0009X,ALFORJA,41.213892,0.963335,406.0,2026-06-05 23:00:00+00:00,15.7,15.7,16.0,0.0,...,1.8,49.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-05
1,0016A,REUS AEROPUERTO,41.145000,1.163611,71.0,2026-06-05 23:00:00+00:00,18.3,18.3,18.4,0.0,...,2.1,260.0,250.0,1008.4,1017.4,27.0,0.0,17.0,15.0,2026-06-05
2,0034X,VALLS,41.293053,1.260838,233.0,2026-06-05 23:00:00+00:00,16.3,16.3,16.7,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-05
3,0042Y,TARRAGONA FAC. GEOGRAFIA,41.123892,1.249167,55.0,2026-06-05 23:00:00+00:00,18.9,18.9,19.2,0.0,...,2.8,242.0,275.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-05
4,0061X,PONTONS,41.417052,1.519269,632.0,2026-06-05 23:00:00+00:00,13.0,13.0,13.2,0.0,...,3.9,299.0,292.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-05


In [4]:
#============================================
# Celda 4 — Análisis rápido AEMET
#============================================
print('Valores nulos por columna:')
nulos = df_aemet.isnull().sum()
print(nulos[nulos > 0])

print(f'\nRango temperatura:  {df_aemet["ta"].min():.1f}°C — {df_aemet["ta"].max():.1f}°C')
print(f'Media temperatura:  {df_aemet["ta"].mean():.1f}°C')
print(f'\nEstaciones con precipitación > 0: {(df_aemet["prec"] > 0).sum()}')
print(f'Precipitación máxima: {df_aemet["prec"].max():.1f} mm')
print(f'\nRango altitud: {df_aemet["alt"].min():.0f}m — {df_aemet["alt"].max():.0f}m')
print(f'\nEstaciones sin coordenadas: {df_aemet[["lat","lon"]].isnull().any(axis=1).sum()}')


Valores nulos por columna:
ta             92
tamin          92
tamax          92
prec          347
hr             85
vv           1271
vmax         1309
dv           1270
dmax         1296
pres         6904
pres_nmar    8204
vis          9058
inso         8556
ts           8610
tpr          7217
dtype: int64

Rango temperatura:  2.4°C — 30.4°C
Media temperatura:  16.9°C

Estaciones con precipitación > 0: 192
Precipitación máxima: 4.8 mm

Rango altitud: 0m — 2856m

Estaciones sin coordenadas: 0


In [5]:
#============================================
# Celda 5 — Explorar estructura WAQI
#============================================
waqi_raw = data['calidad_aire']
ciudades = waqi_raw['ciudades']

print(f'Total ciudades OK:  {waqi_raw["total_ciudades_ok"]}')
print(f'Total errores:      {waqi_raw["total_errores"]}')
print(f'\nKeys de una ciudad:')
print(list(ciudades[0].keys()))
print(f'\nEjemplo ciudad[0]:')
print(json.dumps(ciudades[0], indent=2, ensure_ascii=False))


Total ciudades OK:  30
Total errores:      3

Keys de una ciudad:
['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']

Ejemplo ciudad[0]:
{
  "ciudad": "madrid",
  "nombre": "Madrid",
  "lat": 40.4167754,
  "lon": -3.7037902,
  "timestamp": "2026-06-03 16:00:00",
  "aqi": 53,
  "dominante": "pm25",
  "NO2": 4.2,
  "PM25": 53,
  "PM10": 24,
  "O3": 44.3,
  "SO2": 2.6,
  "CO": 0.1,
  "temperatura": 29.1,
  "humedad": 34.4,
  "presion": 1015.5,
  "viento": 4.8
}


In [6]:
#============================================
# Celda 6 — Parsear WAQI a DataFrame
#============================================
df_waqi = pd.DataFrame(ciudades)

# Convertir tipos
df_waqi['timestamp'] = pd.to_datetime(df_waqi['timestamp'], errors='coerce')

cols_num = ['lat','lon','aqi','NO2','PM25','PM10','O3','SO2','CO',
            'temperatura','humedad','presion','viento']
for c in cols_num:
    if c in df_waqi.columns:
        df_waqi[c] = pd.to_numeric(df_waqi[c], errors='coerce')

print(f'Shape: {df_waqi.shape}')
print(f'Columnas: {list(df_waqi.columns)}')
df_waqi.head(10)


Shape: (30, 17)
Columnas: ['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']


,ciudad,nombre,lat,lon,timestamp,aqi,dominante,NO2,PM25,PM10,O3,SO2,CO,temperatura,humedad,presion,viento
0,madrid,Madrid,40.416775,-3.703790,2026-06-03 16:00:00,53.0,pm25,4.2,53.0,24.0,44.3,2.6,0.1,29.1,34.4,1015.5,4.8
1,barcelona,"Barcelona (Eixample), Catalunya, Spain",41.385343,2.153822,2026-06-06 13:00:00,30.0,pm25,6.9,30.0,15.0,23.6,1.1,0.1,22.7,72.0,1020.9,2.5
2,valencia,"Pista de Silla, València, Valencia, Spain",39.456111,-0.375833,2026-06-03 16:00:00,30.0,pm25,4.2,30.0,16.0,25.6,1.6,0.1,27.7,58.4,NaN,0.1
3,sevilla,"Ranilla, Sevilla, Spain",37.384250,-5.959620,2026-06-06 12:00:00,50.0,pm25,7.4,50.0,24.0,25.8,2.6,0.1,26.3,42.1,1017.4,1.3
4,bilbao,"Mazarredo, Bilbao, País Vasco, Spain",43.267500,-2.935200,2026-06-06 11:00:00,21.0,pm25,1.0,21.0,10.0,NaN,2.6,0.1,18.0,82.0,1022.0,4.1
5,zaragoza,"El Picarral, Zaragoza, Spain",41.670278,-0.871111,2026-06-06 12:00:00,48.0,pm25,2.9,48.0,NaN,29.8,NaN,1.4,24.7,40.0,1020.1,3.0
6,malaga,"Carranque, Malaga, Spain",36.719640,-4.447500,2026-06-06 12:00:00,59.0,pm25,11.0,59.0,37.0,25.4,2.1,0.1,27.0,65.4,1019.6,6.2
7,murcia,"San Basilio Murcia Ciudad, Murcia, Spain",37.993960,-1.144628,2026-06-06 13:00:00,50.0,pm25,2.5,50.0,24.0,34.4,1.0,0.1,28.5,54.3,1018.1,2.8
8,palma,"foners, Baleares, Spain",39.571250,2.657028,2026-06-03 16:00:00,34.0,o3,2.8,34.0,20.0,34.2,2.1,0.1,20.9,66.3,1017.9,13.0
9,alicante,"Rabassa, Alacant, Valencia, Spain",38.351111,-0.513889,2026-06-03 16:00:00,31.0,o3,0.5,13.0,8.0,30.9,0.6,NaN,24.0,71.0,1016.0,2.8


In [7]:
#============================================
# Celda 7 — Análisis rápido WAQI
#============================================
def nivel_aqi(aqi):
    if pd.isna(aqi):       return 'Sin datos'
    elif aqi <= 50:        return '🟢 Bueno'
    elif aqi <= 100:       return '🟡 Moderado'
    elif aqi <= 150:       return '🟠 Insalubre sensibles'
    elif aqi <= 200:       return '🔴 Insalubre'
    else:                  return '🟣 Muy insalubre'

df_waqi['nivel_aqi'] = df_waqi['aqi'].apply(nivel_aqi)

print('Ciudades por nivel AQI:')
print(df_waqi['nivel_aqi'].value_counts().to_string())

print(f'\nTop 5 peor AQI:')
print(df_waqi.nlargest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nTop 5 mejor AQI:')
print(df_waqi.nsmallest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nValores nulos por columna:')
print(df_waqi.isnull().sum()[df_waqi.isnull().sum() > 0])


Ciudades por nivel AQI:
nivel_aqi
🟢 Bueno       25
🟡 Moderado     4
Sin datos      1

Top 5 peor AQI:
                  nombre  aqi dominante  PM25  PM10   O3
Carranque, Malaga, Spain 59.0      pm25  59.0  37.0 25.4
                  Madrid 53.0      pm25  53.0  24.0 44.3
         Erie, Ohio, USA 52.0      pm25  52.0  28.0 32.0
 La Orden, Huelva, Spain 52.0      pm25  52.0  26.0 29.4
 Ranilla, Sevilla, Spain 50.0      pm25  50.0  24.0 25.8

Top 5 mejor AQI:
                                   nombre  aqi dominante  PM25  PM10    O3
Vila Velha - IBES, Espírito Santo, Brazil  8.0        o3   NaN   NaN  8.14
       Santander Centro, Cantabria, Spain 20.0        o3   NaN  16.0 19.90
     Mazarredo, Bilbao, País Vasco, Spain 21.0      pm25  21.0  10.0   NaN
   Arco de ladrillo II, Valladolid, Spain 21.0      pm10  17.0  21.0 23.60
                 Lleida, Catalunya, Spain 23.0        o3  25.0  14.0 22.80

Valores nulos por columna:
aqi        1
NO2        2
PM25       3
PM10       2
O3      

In [8]:
#============================================
# Celda 8 — Resumen calidad de datos
#============================================
resumen = pd.DataFrame([
    {
        'tabla':     'aemet_estaciones',
        'filas':     len(df_aemet),
        'columnas':  len(df_aemet.columns),
        'periodos':  df_aemet['fint'].nunique(),
        'nulos_%':   round(df_aemet.isnull().sum().sum() / (df_aemet.shape[0] * df_aemet.shape[1]) * 100, 1)
    },
    {
        'tabla':     'waqi_ciudades',
        'filas':     len(df_waqi),
        'columnas':  len(df_waqi.columns),
        'periodos':  df_waqi['timestamp'].nunique(),
        'nulos_%':   round(df_waqi.isnull().sum().sum()  / (df_waqi.shape[0]  * df_waqi.shape[1])  * 100, 1)
    },
])
print(resumen.to_string(index=False))


           tabla  filas  columnas  periodos  nulos_%
aemet_estaciones  10253        22        13     24.1
   waqi_ciudades     30        18         9      3.7


In [9]:
#============================================
# Celda 9 — Guardar parquets
#============================================
os.makedirs('output/ambiental/01-raw', exist_ok=True)   # ← sin ../../

df_aemet.to_parquet('output/ambiental/01-raw/raw_aemet_estaciones.parquet', index=False)
print(f'✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet ({len(df_aemet)} filas)')

df_waqi.to_parquet('output/ambiental/raw_waqi_ciudades.parquet', index=False)
print(f'✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet ({len(df_waqi)} filas)')

print('\n🎉 Todos los parquets ambiental guardados en output/')

✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet (10253 filas)
✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet (30 filas)

🎉 Todos los parquets ambiental guardados en output/
